# 04 - Statistical Analysis
In this notebook, we perform statistical tests to validate insights and measure effect sizes to provide strong recommendations.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal
import warnings

warnings.filterwarnings('ignore')

# Helper function for Cramer's V (Effect Size for Chi-Square)
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))


## 1. Load Data


In [ ]:
cleaned_df = pd.read_csv('../data/processed/cleaned_dataset.csv')
cancelled_df = pd.read_csv('../data/processed/cancelled_dataset.csv')  # Included for consistency with EDA

# Create binary column for cancellation tests
cleaned_df['Is_Cancelled'] = cleaned_df['Booking_Status'].apply(lambda x: 0 if x == 'Success' else 1)
print("Data loaded successfully. Total rows:", len(cleaned_df))


## 2. Chi-Square Test & Effect Size: Vehicle Type vs Booking Status
Testing if Vehicle Type significantly impacts whether a ride is cancelled.


In [ ]:
contingency_vehicle = pd.crosstab(cleaned_df['Vehicle_Type'], cleaned_df['Is_Cancelled'])
chi2_v, p_val_v, dof_v, expected_v = chi2_contingency(contingency_vehicle)
effect_size_v = cramers_v(contingency_vehicle)

print(f"Chi-Square Statistic: {chi2_v:.4f}")
print(f"P-value: {p_val_v:.4e}")
print(f"Cramer's V (Effect Size): {effect_size_v:.4f}")

if p_val_v < 0.05:
    print("\nResult: Significant relationship found between Vehicle Type and Cancellation.")
    if effect_size_v < 0.1:
        print("Effect is weak.")
    elif effect_size_v < 0.3:
        print("Effect is moderate.")
    else:
        print("Effect is strong.")


## 3. Mann-Whitney U Test: Driver vs. Customer Cancellations (Ride Distance)
Statistically comparing the ride distances of Driver Cancellations vs Customer Cancellations to inform allocation strategies.


In [ ]:
driver_cancels_dist = cleaned_df[cleaned_df['Booking_Status'] == 'Canceled By Driver']['Ride_Distance'].dropna()
customer_cancels_dist = cleaned_df[cleaned_df['Booking_Status'] == 'Canceled By Customer']['Ride_Distance'].dropna()

stat_mwu, p_val_mwu = mannwhitneyu(driver_cancels_dist, customer_cancels_dist)

# Common Language Effect Size (CLES) approximation for Mann-Whitney
# f = U / (n1 * n2)
cles = stat_mwu / (len(driver_cancels_dist) * len(customer_cancels_dist))

print(f"Mann-Whitney U Statistic: {stat_mwu:.4f}")
print(f"P-value: {p_val_mwu:.4e}")
print(f"Common Language Effect Size: {cles:.4f} (Probability a random Driver Cancel has higher distance than Customer Cancel)")

if p_val_mwu < 0.05:
    print("\nResult: Significant difference in ride distances between Driver and Customer cancellations.")


## 4. Kruskal-Wallis Test: Ride Distance across Vehicle Types
Testing if ride distances differ significantly depending on the vehicle type booked.


In [ ]:
# Group distances by vehicle type
vehicle_types = cleaned_df['Vehicle_Type'].dropna().unique()
distance_groups = [cleaned_df[cleaned_df['Vehicle_Type'] == vt]['Ride_Distance'].dropna() for vt in vehicle_types]

stat_kw, p_val_kw = kruskal(*distance_groups)

print(f"Kruskal-Wallis Statistic: {stat_kw:.4f}")
print(f"P-value: {p_val_kw:.4e}")

if p_val_kw < 0.05:
    print("\nResult: Ride distances vary significantly across different Vehicle Types.")
else:
    print("\nResult: No significant variance in ride distances across Vehicle Types.")


## Conclusion & Summary of Findings

The statistical tests robustly validate our visual EDA findings and establish actionable strength for these relationships:
- **Vehicle Type Impact**: There is a statistically significant relationship between the chosen Vehicle Type and the likelihood of cancellation, though the effect size (Cramer's V) should dictate the urgency of re-allocation strategies.
- **Cancellation Distances**: There is a significant difference in ride distance between Driver Cancellations and Customer Cancellations (validated via Mann-Whitney U), hinting at driver preference or operational limits on long-distance requests.
- **Distance Variance**: The Kruskal-Wallis test confirms that booked ride distances vary significantly across vehicle types, meaning vehicle allocation is highly distance-dependent and plays heavily into optimization strategies.
